In [1]:
# /// script
# require-python = ">=3.11"
# dependencies = ["duckdb", "ipywidgets", "jupysql", "matplotlib", "numpy", "pandas", "tqdm", "zstandard"]
# ///

# _Dirty_ labels of normal activity

Earlier embedding work for the ACME4 dataset got me to hand-label a few data clusters.
Further eyeballing revealed that these clusters were much less uniform in content than I thought,
to the point where I now consider these labels _dirty_, and thus rather weak.
Nevertheless, here they are.

In [2]:
import duckdb
import logging as lg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm, trange
import warnings

In [3]:
tqdm.pandas()

In [4]:
lg.basicConfig(level=lg.INFO, format="%(asctime)-8s | %(levelname)-8s | %(name)-24s | %(message)s", datefmt="%H:%M:%S")
LOG = lg.getLogger("notebook")

In [5]:
pd.set_option("display.html.use_mathjax", False)
pd.set_option("display.max_colWidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.min_rows", 20)
pd.set_option("styler.html.mathjax", False)

In [6]:
root_acme4 = Path("/data/ACME4/stdview-20240819-20240923")  # Replace with your own data location!
db = duckdb.connect()
for path in root_acme4.glob("*.parquet"):
    name_view = path.with_suffix("").name
    LOG.debug(f"Add view {name_view} over {path}")
    db.sql(f"create or replace view {name_view} as select * from '{path}'")

In [7]:
with warnings.catch_warnings(action="ignore", category=SyntaxWarning):
    %load_ext sql
%sql db --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [8]:
%%sql
create or replace view labels_dirty as select * from 'labels-dirty.parquet'

,Count


In [13]:
%%sql
select process.pid_hash, process_name, args, label
from process
inner join labels_dirty using (pid_hash)
order by pid_hash

,pid_hash,process_name,args,label
0,00000CE18D05E89D614D0C0C0E6F271B,mergehelper.exe,c:\programdata\wintap\parquet\focuschange_sensor 133687977291137033,Wintap mergehelper tool
1,000056E84BA54E1365F010F0DAE54B75,wmic.exe,os get version /format:list,WBEM/WMI
2,00006CA11E4E7EDFB2CB3064DF6A5A04,wmic.exe,computersystem get domain /value,WBEM/WMI
3,000078C1DB396855E217442A15EDBDC6,mergehelper.exe,c:\programdata\wintap\parquet\default_sensor\microsoftwindowsgrouppolicy 133691487619222664,Wintap mergehelper tool
4,0000C6AC5D07B0A125832B07E4224077,backgroundtaskhost.exe,-servername:backgroundtaskhost.webaccountprovider,Authentication services
5,00011CC17AD656E8A5437991DFC034BD,mousocoreworker.exe,-embedding,Windows Update services
6,00017CCD682C2EA4B3EC4E192F009FEC,wmic.exe,os get caption /format:list,WBEM/WMI
7,0001B7E300B421A7FF808A2625B9247E,mergehelper.exe,c:\programdata\wintap\parquet\default_sensor\wmiactivity 133690365717520306,Wintap mergehelper tool
8,00020C36D162EB89E967897ED01BB82A,wmic.exe,os get caption /format:list,WBEM/WMI
9,0002427D820AB07D8CFB99D1DDC0EEB9,wmic.exe,computersystem get domain /value,WBEM/WMI
